# GWM-RNN Training on Kaggle - WN18RR Dataset with **Hybrid Embeddings**

Train the lightweight GWM-RNN model for Knowledge Graph Completion on WN18RR.

**Dataset: WN18RR**
- ~40,943 entities (WordNet synsets)
- 11 relation types
- ~86,836 training triples
- **Average neighbors: 8.56** (sparse graph - no filtering needed)
- **Task**: Given `(head, relation, ?)`, predict the tail entity

**Hybrid Embeddings Architecture:**
- **BERT embeddings** (768D): Semantic similarity from text descriptions (frozen)
- **Learnable embeddings** (768D): Geometric patterns from graph structure (trained)
- **Combined**: 1536D input = Better entity disambiguation + structural precision
---

## 1. Install Dependencies

In [ ]:
import os
import sys

# Check environment
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Running on Kaggle: {IS_KAGGLE}")

if IS_KAGGLE:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("✓ Environment setup complete")

In [ ]:
# Install required packages
!pip install -q torch scikit-learn tqdm matplotlib seaborn

print("✓ All dependencies installed")

## 2. Configuration

Configure paths and training parameters for WN18RR. We'll train multiple configurations:
- **3 Pooling Methods**: last, mean, max
- **Multiple Hyperparameter Sets**: Optimized for WN18RR characteristics
- **2 Loss Functions**: InfoNCE (contrastive) and Margin Ranking

In [ ]:
# ==============================================================================
# DATA PATHS CONFIGURATION
# ==============================================================================
if IS_KAGGLE:
    # Kaggle input paths for WN18RR dataset
    DATA_DIR = '/kaggle/input/gwm-rnn-kg-wn18rr'  # Your processed WN18RR data
    OUTPUT_BASE_DIR = '/kaggle/working/experiments'
else:
    # Local paths
    DATA_DIR = 'D:/NLP research/Code/graph-world-models/GWM/data/wn18rr/processed/relation-prediction'
    OUTPUT_BASE_DIR = './trained/gwm-rnn/wn18rr/experiments'

# ==============================================================================
# POOLING METHODS TO TEST
# ==============================================================================
POOLING_METHODS = ['last', 'mean', 'max']

# ==============================================================================
# HYPERPARAMETER CONFIGURATIONS (Optimized for WN18RR)
# ==============================================================================

HYPERPARAMETER_SETS = [
    # Standard configuration (balanced performance)
    {
        'name': 'standard',
        'hidden_dim': 512,
        'num_lstm_layers': 2,
        'dropout': 0.1,
        'learning_rate': 1e-3,
        'batch_size': 256,
        'num_negatives': 10,
        'loss': 'infonce',
        'temperature': 0.07,
        'use_in_batch_negatives': False,
        'learnable_dim': 768,
        'hybrid_weight': 0.5,
        'description': 'Standard config optimized for WN18RR with hybrid embeddings'
    },
    # In-batch negatives (efficient for large entity space)
    {
        'name': 'in-batch',
        'hidden_dim': 512,
        'num_lstm_layers': 2,
        'dropout': 0.1,
        'learning_rate': 1e-3,
        'batch_size': 512,
        'num_negatives': 0,
        'loss': 'infonce',
        'temperature': 0.07,
        'use_in_batch_negatives': True,
        'learnable_dim': 768,
        'hybrid_weight': 0.5,
        'description': 'In-batch negatives (511 per sample) + hybrid embeddings for large entity space'
    },
    # Larger capacity (handle 40k+ entities)
    {
        'name': 'large',
        'hidden_dim': 1024,
        'num_lstm_layers': 2,
        'dropout': 0.15,
        'learning_rate': 5e-4,
        'batch_size': 2048,
        'num_negatives': 0,
        'loss': 'infonce',
        'temperature': 0.07,
        'use_in_batch_negatives': True,
        'learnable_dim': 768,
        'hybrid_weight': 0.5,
        'description': 'Larger hidden dim + hybrid embeddings for 40k+ entity space'
    },
    # Conservative (prevent overfitting on sparse graph)
    {
        'name': 'conservative',
        'hidden_dim': 512,
        'num_lstm_layers': 2,
        'dropout': 0.2,
        'learning_rate': 5e-4,
        'batch_size': 512,
        'num_negatives': 0,
        'loss': 'infonce',
        'temperature': 0.1,
        'use_in_batch_negatives': True,
        'learnable_dim': 768,
        'hybrid_weight': 0.5,
        'description': 'Conservative config + hybrid embeddings with regularization for sparse graph'
    },
]

# ==============================================================================
# TRAINING PARAMETERS (FIXED ACROSS ALL EXPERIMENTS)
# ==============================================================================
NUM_EPOCHS = 500
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 15
SCHEDULER_PATIENCE = 7
EVAL_EVERY = 1
SEED = 42
NUM_WORKERS = 2

# ==============================================================================
# EXPERIMENT SELECTION
# ==============================================================================
RUN_ALL_CONFIGS = True
SELECTED_CONFIG = 'standard'

print("="*80)
print(" "*15 + "GWM-RNN EXPERIMENT CONFIGURATION - WN18RR")
print("="*80)
print(f"\n📊 Dataset: WN18RR (WordNet Knowledge Graph)")
print(f"   ~40,943 entities (WordNet synsets)")
print(f"   11 relation types (22 with inverses)")
print(f"   ~86,836 training triples")
print(f"   Average neighbors per entity: 8.56 (sparse - pristine structure)")
print(f"   Task: Knowledge Graph Completion")

print(f"\n📁 Data Directory: {DATA_DIR}")
print(f"📁 Output Base Directory: {OUTPUT_BASE_DIR}")

print(f"\n🔄 Pooling Methods to Test ({len(POOLING_METHODS)}):")
for pooling in POOLING_METHODS:
    print(f"   • {pooling}")

print(f"\n⚙️  Hyperparameter Sets Available ({len(HYPERPARAMETER_SETS)}):")
for i, config in enumerate(HYPERPARAMETER_SETS, 1):
    print(f"   {i}. {config['name']:15s} - {config['description']}")
    neg_info = f"In-batch ({config['batch_size']-1} negatives)" if config.get('use_in_batch_negatives', False) else f"{config['num_negatives']} sampled"
    print(f"      Hidden: {config['hidden_dim']}, Layers: {config['num_lstm_layers']}, "
          f"Loss: {config['loss']}, Negatives: {neg_info}")

if RUN_ALL_CONFIGS:
    print(f"\n🚀 Mode: Running ALL configurations")
    print(f"   Total experiments: {len(POOLING_METHODS)} pooling × {len(HYPERPARAMETER_SETS)} configs = {len(POOLING_METHODS) * len(HYPERPARAMETER_SETS)} experiments")

else:
    print(f"\n🎯 Mode: Running SELECTED configuration only")
    print(f"   Config: {SELECTED_CONFIG}")
    print(f"   Total experiments: {len(POOLING_METHODS)} pooling × 1 config = {len(POOLING_METHODS)} experiments")


print(f"\n⏱️  Fixed Parameters:")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Gradient clipping: {MAX_GRAD_NORM}")
print(f"   Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"   Seed: {SEED}")

print("="*80)

## 3. Copy Training Files from GitHub

Clone repository and copy training scripts for relation prediction.

In [ ]:
required_files = ['model.py', 'dataset.py', 'inference.py', 'train.py', 'utils.py', 'generate_negatives.py']

if IS_KAGGLE:
    print("="*70)
    print("Cloning GitHub repository...")
    print("="*70)
    
    # Clone your GitHub repo
    GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
    BRANCH = "context-aware-gwm-rnn"
    
    !git clone {GITHUB_REPO} /kaggle/working/gwm
    %cd /kaggle/working/gwm
    !git checkout {BRANCH}
    !git pull
    %cd ../
    
    # Copy files from repo to working directory
    repo_path = "/kaggle/working/gwm/gwm-rnn/relation-prediction"
    
    print(f"\nCopying files from {repo_path}...")
    for file in required_files:
        !cp {repo_path}/{file} /kaggle/working/
        print(f"✓ Copied {file}")
else:
    print("Running locally - files should be in current directory")

# Verify files exist
import os
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"\n❌ Missing files: {missing_files}")
    raise FileNotFoundError(f"Required files not found: {missing_files}")
else:
    print(f"\n✓ All required files ready: {required_files}")

## 3.5 Generate Fixed Negative Samples

**Important**: Generate fixed negative samples to ensure fair comparison across all model variants.

All models will use the same negative samples, eliminating variance from random sampling.

In [ ]:
import torch
from pathlib import Path

# On Kaggle, save to working directory (input is read-only)
if IS_KAGGLE:
    negatives_output_dir = Path('/kaggle/working')
else:
    negatives_output_dir = Path(DATA_DIR)

negatives_path = negatives_output_dir / 'train_negatives.pt'
data_dir_negatives = Path(DATA_DIR) / 'train_negatives.pt'

# Determine max negatives needed
max_negatives = max(config['num_negatives'] for config in HYPERPARAMETER_SETS)
print(f"ℹ️  Maximum negatives needed: {max_negatives}")

if negatives_path.exists():
    print("=" * 70)
    print("✓ Fixed negatives already exist")
    print("=" * 70)
    print(f"Location: {negatives_path}")
    
    train_negatives = torch.load(negatives_path, map_location='cpu')
    print(f"Shape: {train_negatives.shape}")
    print(f"Size: {negatives_path.stat().st_size / (1024**2):.2f} MB")
    
    if train_negatives.shape[1] < max_negatives:
        print(f"\n⚠️  WARNING: Need {max_negatives} negatives, regenerating...")
        !python generate_negatives.py \
            --data_dir {DATA_DIR} \
            --output_dir {negatives_output_dir} \
            --num_negatives {max_negatives} \
            --seed {SEED} \
            --force
    else:
        print("\nAll experiments will use these identical negative samples.")
        print("=" * 70)
elif data_dir_negatives.exists():
    print("=" * 70)
    print("✓ Fixed negatives found in dataset")
    print("=" * 70)
    print(f"Location: {data_dir_negatives}")
    
    train_negatives = torch.load(data_dir_negatives, map_location='cpu')
    print(f"Shape: {train_negatives.shape}")
    
    if train_negatives.shape[1] < max_negatives:
        print(f"\n⚠️  Need {max_negatives} negatives, regenerating...")
        !python generate_negatives.py \
            --data_dir {DATA_DIR} \
            --output_dir {negatives_output_dir} \
            --num_negatives {max_negatives} \
            --seed {SEED} \
            --force
    else:
        import shutil
        shutil.copy(data_dir_negatives, negatives_path)
        print(f"Copied to: {negatives_path}")
        print("=" * 70)
else:
    print("=" * 70)
    print("GENERATING FIXED NEGATIVE SAMPLES (WN18RR)")
    print("=" * 70)
    print(f"\nGenerating {max_negatives} negatives per triple...")
    
    !python generate_negatives.py \
        --data_dir {DATA_DIR} \
        --output_dir {negatives_output_dir} \
        --num_negatives {max_negatives} \
        --seed {SEED}
    
    print("=" * 70)
    print("GENERATION COMPLETE")
    print("=" * 70)

## 4. Run Training Experiments

Train models with all pooling methods and selected hyperparameter configurations on WN18RR.

**Note**: All models use the same pre-generated negative samples and context embeddings for fair comparison.

In [ ]:
import time
from datetime import datetime
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Determine which configs to run
if RUN_ALL_CONFIGS:
    configs_to_run = HYPERPARAMETER_SETS
else:
    configs_to_run = [c for c in HYPERPARAMETER_SETS if c['name'] == SELECTED_CONFIG]

# Track all experiment results
all_results = []
experiment_start_time = time.time()

print("="*80)
print(" "*20 + "STARTING WN18RR EXPERIMENTS")
print("="*80)
print(f"\nTotal experiments to run: {len(POOLING_METHODS)} × {len(configs_to_run)} = {len(POOLING_METHODS) * len(configs_to_run)}")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

experiment_num = 0
total_experiments = len(POOLING_METHODS) * len(configs_to_run)

for config in configs_to_run:
    for pooling in POOLING_METHODS:
        experiment_num += 1
        
        print(f"\n{'='*80}")
        print(f" EXPERIMENT {experiment_num}/{total_experiments}: {config['name'].upper()} + {pooling.upper()}-POOLING (WN18RR)")
        print(f"{'='*80}")
        
        # Create output directory for this experiment
        output_dir = f"{OUTPUT_BASE_DIR}/{config['name']}/{pooling}-pooling"
        
        # Build training command
        if config['loss'] == 'infonce':
            loss_args = f"--loss infonce --temperature {config.get('temperature', 0.07)}"
            if config.get('use_in_batch_negatives', False):
                loss_args += " --use_in_batch_negatives"
        else:
            loss_args = f"--loss margin --margin {config.get('margin', 1.0)}"
        
        cmd = f"""python train.py \\
            --data_dir {DATA_DIR} \\
            --output_dir {output_dir} \\
            --hidden_dim {config['hidden_dim']} \\
            --num_lstm_layers {config['num_lstm_layers']} \\
            --dropout {config['dropout']} \\
            --pooling {pooling} \\
            --num_epochs {NUM_EPOCHS} \\
            --batch_size {config['batch_size']} \\
            --learning_rate {config['learning_rate']} \\
            --weight_decay {WEIGHT_DECAY} \\
            --max_grad_norm {MAX_GRAD_NORM} \\
            --num_negatives {config['num_negatives']} \\
            {loss_args} \\
            --learnable_dim {config['learnable_dim']} \\
            --hybrid_weight {config['hybrid_weight']} \\
            --scheduler_patience {SCHEDULER_PATIENCE} \\
            --early_stopping_patience {EARLY_STOPPING_PATIENCE} \\
            --eval_every {EVAL_EVERY} \\
            --seed {SEED} \\
            --num_workers {NUM_WORKERS}"""
        
        print(f"\n📋 Configuration:")
        print(f"   Dataset: WN18RR (~40,943 entities, 11 relations)")
        print(f"   Config: {config['name']} - {config['description']}")
        print(f"   Pooling: {pooling}")
        print(f"   Hidden dim: {config['hidden_dim']}")
        print(f"   LSTM layers: {config['num_lstm_layers']}")
        print(f"   Dropout: {config['dropout']}")
        print(f"   Loss: {config['loss']}")
        if config.get('use_in_batch_negatives', False):
            print(f"   Negatives: In-batch ({config['batch_size']-1} per sample)")
        else:
            print(f"   Negatives: {config['num_negatives']} sampled per sample")
        print(f"   Learnable dim: {config['learnable_dim']} (hybrid embeddings)")
        print(f"   Hybrid weight: {config['hybrid_weight']} (BERT/learnable balance)")
        print(f"   Output: {output_dir}")
        
        print(f"\n🚀 Starting training...")
        print("-"*80)
        
        # Execute training
        !{cmd}
        
        # Load and store results
        try:
            result_path = Path(output_dir) / "test_results.json"
            history_path = Path(output_dir) / "training_history.json"
            
            if result_path.exists() and history_path.exists():
                with open(result_path) as f:
                    test_results = json.load(f)
                with open(history_path) as f:
                    history = json.load(f)
                
                test_metrics = test_results['test_metrics']
                train_time = sum(history['epoch_times'])
                
                all_results.append({
                    'config_name': config['name'],
                    'pooling': pooling,
                    'hidden_dim': config['hidden_dim'],
                    'num_layers': config['num_lstm_layers'],
                    'dropout': config['dropout'],
                    'loss': config['loss'],
                    'learning_rate': config['learning_rate'],
                    'batch_size': config['batch_size'],
                    'learnable_dim': config['learnable_dim'],
                    'hybrid_weight': config['hybrid_weight'],
                    'test_mrr': test_metrics['MRR'],
                    'test_mr': test_metrics['MR'],
                    'test_hits@1': test_metrics['Hits@1'],
                    'test_hits@3': test_metrics['Hits@3'],
                    'test_hits@10': test_metrics['Hits@10'],
                    'test_hits@50': test_metrics['Hits@50'],
                    'best_val_mrr': test_results['best_val_mrr'],
                    'best_epoch': test_results['best_epoch'],
                    'training_time': train_time,
                    'output_dir': output_dir
                })
                
                print(f"\n✅ Experiment {experiment_num} completed successfully!")
                print(f"   Test MRR: {test_metrics['MRR']:.4f}")
                print(f"   Test Hits@10: {test_metrics['Hits@10']:.4f} ({test_metrics['Hits@10']*100:.2f}%)")
                print(f"   Training time: {train_time:.1f}s ({train_time/60:.1f} min)")
            else:
                print(f"\n⚠️  Results not found for experiment {experiment_num}")
        except Exception as e:
            print(f"\n❌ Error loading results for experiment {experiment_num}: {e}")
        
        print(f"\n{'='*80}\n")

total_time = time.time() - experiment_start_time

print(f"\n{'='*80}")
print(" "*20 + "ALL WN18RR EXPERIMENTS COMPLETED")
print(f"{'='*80}")
print(f"Total experiments: {len(all_results)}/{total_experiments}")
print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
if len(all_results) > 0:
    print(f"Average time per experiment: {total_time/len(all_results)/60:.1f} minutes")

print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}\n")

## 5. Comprehensive Results Analysis

Analyze and compare all WN18RR experiments across pooling methods and hyperparameter sets.

In [ ]:
if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    print("="*80)
    print(" "*15 + "WN18RR EXPERIMENT RESULTS SUMMARY")
    print("="*80)
    
    # Overall best
    best_idx = df_results['test_mrr'].idxmax()
    best_result = df_results.loc[best_idx]
    
    print(f"\n🏆 BEST OVERALL PERFORMANCE:")
    print(f"   Config: {best_result['config_name']} + {best_result['pooling']}-pooling")
    print(f"   Test MRR: {best_result['test_mrr']:.4f}")
    print(f"   Test Hits@1: {best_result['test_hits@1']:.4f} ({best_result['test_hits@1']*100:.2f}%)")
    print(f"   Test Hits@10: {best_result['test_hits@10']:.4f} ({best_result['test_hits@10']*100:.2f}%)")
    print(f"   Test MR: {best_result['test_mr']:.2f}")
    print(f"   Training time: {best_result['training_time']:.1f}s ({best_result['training_time']/60:.1f} min)")
    
    # Pooling method comparison
    print(f"\n📊 PERFORMANCE BY POOLING METHOD:")
    print("-"*80)
    pooling_summary = df_results.groupby('pooling').agg({
        'test_mrr': ['mean', 'std', 'max'],
        'test_hits@10': ['mean', 'max'],
        'test_mr': ['mean', 'min'],
        'training_time': 'mean'
    }).round(4)
    
    for pooling in POOLING_METHODS:
        if pooling in pooling_summary.index:
            row = pooling_summary.loc[pooling]
            print(f"\n{pooling.upper()}-Pooling:")
            print(f"   Avg MRR: {row[('test_mrr', 'mean')]:.4f} ± {row[('test_mrr', 'std')]:.4f}")
            print(f"   Max MRR: {row[('test_mrr', 'max')]:.4f}")
            print(f"   Avg Hits@10: {row[('test_hits@10', 'mean')]:.4f}")
            print(f"   Avg MR: {row[('test_mr', 'mean')]:.2f}")
            print(f"   Avg Training Time: {row[('training_time', 'mean')]/60:.1f} min")
    
    # Configuration comparison
    if len(configs_to_run) > 1:
        print(f"\n⚙️  PERFORMANCE BY CONFIGURATION:")
        print("-"*80)
        config_summary = df_results.groupby('config_name').agg({
            'test_mrr': ['mean', 'std', 'max'],
            'test_hits@10': ['mean', 'max'],
            'training_time': 'mean'
        }).round(4)
        
        for config_name in [c['name'] for c in configs_to_run]:
            if config_name in config_summary.index:
                row = config_summary.loc[config_name]
                config_desc = [c['description'] for c in configs_to_run if c['name'] == config_name][0]
                print(f"\n{config_name.upper()} ({config_desc}):")
                print(f"   Avg MRR: {row[('test_mrr', 'mean')]:.4f} ± {row[('test_mrr', 'std')]:.4f}")
                print(f"   Max MRR: {row[('test_mrr', 'max')]:.4f}")
                print(f"   Avg Hits@10: {row[('test_hits@10', 'mean')]:.4f}")
                print(f"   Avg Training Time: {row[('training_time', 'mean')]/60:.1f} min")
    
    # Detailed results table
    print(f"\n📋 DETAILED RESULTS TABLE:")
    print("-"*80)
    display_cols = ['config_name', 'pooling', 'test_mrr', 'test_hits@1', 'test_hits@10', 
                    'test_mr', 'hidden_dim', 'loss', 'training_time']
    df_display = df_results[display_cols].copy()
    df_display['test_mrr'] = df_display['test_mrr'].apply(lambda x: f"{x:.4f}")
    df_display['test_hits@1'] = df_display['test_hits@1'].apply(lambda x: f"{x:.4f}")
    df_display['test_hits@10'] = df_display['test_hits@10'].apply(lambda x: f"{x:.4f}")
    df_display['test_mr'] = df_display['test_mr'].apply(lambda x: f"{x:.2f}")
    df_display['training_time'] = df_display['training_time'].apply(lambda x: f"{x/60:.1f}min")
    
    print(df_display.to_string(index=False))
    
    # Save results
    results_csv_path = f"{OUTPUT_BASE_DIR}/wn18rr_all_results_summary.csv"
    df_results.to_csv(results_csv_path, index=False)
    print(f"\n✓ Saved detailed results to: {results_csv_path}")
    
    print("\n" + "="*80)
else:
    print("❌ No results available. Please run training experiments first.")